# scPRINT test run

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import os
import pickle
import torch
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
from scanpy.plotting.palettes import zeileis_28
from tqdm.contrib.concurrent import *
from tqdm.auto import *
import anndata
import scanpy as sc
import statistics as stat
import json
import csv
import re
import copy
from sklearn.preprocessing import OneHotEncoder

/scratch/users/k25055720/conda_envs/scprint/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["SCPRINTER_DATA"] = "/scratch/users/k25055720/.cache/scprint"

import snapatac2 as snap
import scprinter as scp

print("snapatac2:", snap.__version__)
print("scprinter:", scp.__version__)

snapatac2: 2.8.0
scprinter: 1.2.1.dev11+geedc9de75


In [3]:
import sys
print(sys.executable)
!nvidia-smi

/scratch/users/k25055720/conda_envs/scprint/bin/python
Fri Feb 13 11:40:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200                    On  |   00000000:19:00.0 Off |                    0 |
| N/A   35C    P0             76W /  700W |       0MiB / 143771MiB |      0%      Default |
|                                         |                        |             Disa

In [ ]:
# Specify directories
main_dir = '/scratch/prj/stem_cells_pituitary'
atac_dir = f'{main_dir}/georgia_atac'
out_dir = f'{main_dir}/Georgia/scPRINT/test_GSM4594389'

In [ ]:
# Using one split lactotroph file first for trial
# Load the lactotroph fragment data 
fragment_file = f'{atac_dir}/mouse/GSM4594389/GSM4594389_Female_pit_2_fragments_sorted.tsv.gz'

In [ ]:
fragment_file

In [ ]:
# Specify genome

# In terminal:
# wget https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_mouse/release_M25/gencode.vM25.basic.annotation.gff3.gz
# gunzip gencode.vM25.basic.annotation.gff3.gz
# mv gencode.vM25.basic.annotation.gff3 gencode_vM25_GRCm38.gff3

genome = scp.genome.mm10
#gff_file = "/scratch/users/k25055720/.cache/scprint/gencode_vM25_GRCm38.gff3"

In [ ]:
# Initialize the scPrinter object
start = time.time()
import time
printer_path = os.path.join(out_dir, 'GSM4594389.h5ad')
if os.path.exists(printer_path):
    printer = scp.load_printer(f'{out_dir}/GSM4594389.h5ad', genome)
else:
    # Load data from the fragments file. Needs at least theses four columns: chromosome, start, end, cell barcode
    printer = scp.pp.import_fragments(
                            path_to_frags=f'{fragment_file}',
                            barcodes=None,
                            savename=printer_path,
                            genome=genome,
                            min_num_fragments=1000, 
                            min_tsse=7,
                            sorted_by_barcode=False,
                            low_memory=True,
                            )

In [ ]:
# When you finish using the object, run printer.close() otherwise you won't be able to load it properly next time.
# This takes ages lowkey eek 
printer.close()
# printer = scp.load_printer(f'{out_dir}/GSM4594389_Lactotrophs.h5ad', genome)

In [ ]:
# Check object 
printer

### Peak Calling

The preset "seq2PRINT" generates peaks that is appropriate to train a seq2PRINT model on.

It's recommended to do this instead of a personal predefined peak region to train seq2PRINT.